# Analisi emotiva di un singolo commento con ELIta

Questo notebook come il **metodo finale** (_lessico ELIta ibrido (α=0.5) + corpus_mean normalisation (Formula 3.5 ItEm)_) assegna un'emozione a un commento del corpus `r/Italia: notizie, film, sport`.

## Setup e caricamento dati

In [29]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from Fase3.support import (BASIC_EMOTIONS, POS_FILTER, load_corpus, compute_mu_e,
                           normalizza, emozione_dominante, plot_radar_single, plot_raw_vs_norm_bars, load_elita_matrix, )
#!pip install dataframe_image
import dataframe_image as dfi

CORPUS_CSV   = Path('corpus_Italia_multi.csv')
TOKENS_CSV   = Path('tokens_Italia_multi.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')

print('Configurazione caricata.')

Configurazione caricata.


In [30]:
df_corpus, df_tokens = load_corpus(CORPUS_CSV, TOKENS_CSV)
df_elita_final = load_elita_matrix(ALPHA_05_CSV)
mu_e = compute_mu_e(df_corpus, df_tokens, df_elita_final)

print('Corpus:', len(df_corpus))
print('Token:', len(df_tokens))
print('Post:', (df_corpus['type'] == 'post').sum())
print('Commenti:', (df_corpus['type'] == 'comment').sum())

Corpus: 4599
Token: 175371
Post: 300
Commenti: 4299


In [31]:
print(df_tokens['pos'].value_counts().to_string())

pos
NOUN     35295
ADP      23932
VERB     23474
DET      18074
ADV      16180
PRON     12518
ADJ      11612
AUX      10060
PROPN     7483
CCONJ     7008
SCONJ     4619
NUM       2329
X         1656
PUNCT      413
INTJ       403
EMOJI      166
SYM        120
PART        29


## Selezione del documento

In [32]:
DOC_INDEX = 4118       # valore tra (0-5001)
DOC_ID    = None       # oppure specifica un ID, es. 'kfr4gvl' (commento) o '1idmjsb' (post)
DOC_TYPE  = 'comment'  # 'comment', 'post', oppure None per tutti

df_sel = df_corpus[df_corpus['type'] == DOC_TYPE] if DOC_TYPE else df_corpus # Filtra per tipo se richiesto
df_sel = df_sel.reset_index(drop=True)

if DOC_ID:
    row = df_sel[df_sel['doc_id'] == DOC_ID].iloc[0]
else:
    row = df_sel.iloc[DOC_INDEX]

CID  = row['doc_id']
TEXT = row['text']

print(f'ID          : {CID}')
print(f'Tipo        : {row["type"]}')
print(f'Autore      : {row["author"]}')
print(f'Score Reddit: {row["score"]}')
print()
print('Testo:')
print('-' * 100)
print(TEXT)
print('-' * 100)

ID          : koxzqfz
Tipo        : comment
Autore      : marco0124
Score Reddit: 3

Testo:
----------------------------------------------------------------------------------------------------
Condivido la mia esperienza. 

Anche io ho smesso di fare sport per 7-8 anni. 
Sono arrivato al punto di odiare il mio fisico (ero sovrappeso) e ho iniziato ad allenarmi in palestra con un mio amico. 
Ho letto che palestra non fa per te. L'importante secondo me è crearsi la propria routine; all'inizio sarà difficile ma vedrai che dopo un paio di mesi risulterà normale. 

Ti consiglio di buttarti e non tergiversare perché si trova sempre un motivo per non iniziare. 
Ti piace boxe? Vai buttati! La soddisfazione che ti dà allenarti e vedere i risultati è inappagabile!

Buona fortuna e daje forte 💪
----------------------------------------------------------------------------------------------------


## Token e lemmi del commento

In [33]:
df_tok = df_tokens[df_tokens['doc_id'] == CID].copy()
elita_idx_all = set(df_elita_final.index)

df_tok['in_ELIta'] = df_tok['lemma'].isin(elita_idx_all)
df_tok['pos_ok']   = df_tok['pos'].isin(POS_FILTER)
df_tok['usato']    = df_tok['in_ELIta'] & df_tok['pos_ok']

print(f'Token totali nel documento          : {len(df_tok)}')
print(f'Con POS valida (ADJ/NOUN/VERB/AUX/EMOJI): {df_tok["pos_ok"].sum()}')
print(f'Trovati in ELIta                    : {df_tok["in_ELIta"].sum()}')
print(f'Token usati per l\'analisi           : {df_tok["usato"].sum()}')
print()

display(df_tok[['token','lemma','pos','in_ELIta','usato']].reset_index(drop=True))

Token totali nel documento          : 107
Con POS valida (ADJ/NOUN/VERB/AUX/EMOJI): 82
Trovati in ELIta                    : 47
Token usati per l'analisi           : 44



,token,lemma,pos,in_ELIta,usato
0,Condivido,condivido,VERB,False,False
1,la,il,DET,False,False
2,mia,mio,DET,True,False
3,esperienza,esperienza,NOUN,True,True
4,Anche,anche,ADV,False,False
...,...,...,...,...,...
102,fortuna,fortuna,NOUN,True,True
103,e,e,CCONJ,False,False
104,daje,daje,NOUN,False,False
105,forte,forte,ADJ,True,True


Anche se ELIta è descritta come composta da aggettivi, nomi e verbi, in realtà contiene anche altri POS (come AUX e EMOJI) che possono contribuire all'analisi emotiva. Per questo motivo, consideriamo utili tutti i token che appartengono a POS validi e sono presenti in ELIta.

## Contributo emotivo per parola

Per ogni lemma usato nell'analisi, mostriamo il vettore emotivo da ELIta α=0.5 (score grezzi).

In [34]:
lemmi_usati = df_tok[df_tok['usato']]['lemma'].tolist()

if not lemmi_usati:
    print('Nessun lemma utile trovato in questo commento.')
else:
    contrib_rows = []
    for lemma in lemmi_usati:
        scores = df_elita_final.loc[lemma, BASIC_EMOTIONS].to_dict()
        dom    = max(scores, key=scores.get)
        scores['lemma']   = lemma
        scores['dom_emo'] = dom
        contrib_rows.append(scores)

    df_contrib = pd.DataFrame(contrib_rows)
    cols_show  = ['lemma'] + BASIC_EMOTIONS + ['dom_emo']

    totals  = df_contrib[BASIC_EMOTIONS].sum()
    dom_raw = totals.idxmax()

    print('Contributi emotivi per lemma (ELIta α=0.5 — score grezzi):')
    display(
        df_contrib[cols_show]
        .style
        .background_gradient(subset=BASIC_EMOTIONS, cmap='YlOrRd', axis=None)
        .format({e: '{:.2f}' for e in BASIC_EMOTIONS})
    )
    styled = (
    df_contrib[cols_show]
    .style
    .background_gradient(subset=BASIC_EMOTIONS, cmap='YlOrRd', axis=None)
    .format({e: '{:.2f}' for e in BASIC_EMOTIONS})
    )
dfi.export(styled, 'tabella_contributi.png', table_conversion='matplotlib', dpi=200)

Contributi emotivi per lemma (ELIta α=0.5 — score grezzi):


,lemma,gioia,aspettativa,rabbia,disgusto,tristezza,sorpresa,paura,fiducia,dom_emo
0,esperienza,0.80,0.93,0.15,0.13,0.32,0.47,0.43,0.64,aspettativa
1,avere,0.64,0.69,0.35,0.16,0.22,0.49,0.57,0.71,fiducia
2,smettere,0.19,0.32,0.67,0.30,0.53,0.26,0.40,0.19,rabbia
3,fare,0.49,0.77,0.07,0.05,0.09,0.34,0.15,0.55,aspettativa
4,sport,0.58,0.49,0.41,0.17,0.18,0.44,0.21,0.44,gioia
5,anno,0.48,0.73,0.17,0.11,0.38,0.41,0.33,0.58,aspettativa
6,arrivare,0.80,0.87,0.15,0.12,0.24,0.70,0.37,0.75,aspettativa
7,punto,0.17,0.52,0.21,0.10,0.20,0.45,0.17,0.30,aspettativa
8,odiare,0.08,0.29,0.96,0.91,0.56,0.37,0.62,0.18,rabbia
9,fisico,0.49,0.59,0.22,0.28,0.36,0.41,0.38,0.46,aspettativa


/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/dataframe_image/converter/matplotlib_table.py:155: UserWarning:

Glyph 128170 (\N{FLEXED BICEPS}) missing from font(s) DejaVu Sans.

/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/dataframe_image/converter/matplotlib_table.py:328: UserWarning:

Glyph 128170 (\N{FLEXED BICEPS}) missing from font(s) DejaVu Sans.



In [35]:
print('Score aggregato grezzo (S_e):')
print(totals.round(3).to_string())
print(f'\n=> Emozione dominante (raw): {dom_raw.upper()}')

Score aggregato grezzo (S_e):
gioia          24.764
aspettativa    28.516
rabbia         12.971
disgusto        9.343
tristezza      13.091
sorpresa       20.471
paura          15.760
fiducia        25.139

=> Emozione dominante (raw): ASPETTATIVA


## Corpus_mean normalisation (Formula 3.5 ItEm)

Lo score grezzo viene diviso per la media di corpus per ogni emozione (μ_e), ottenendo lo score normalizzato (S_e_norm). Le emozioni con valore medio alto nel corpus (come aspettativa) vengono penalizzate proporzionalmente.

In [36]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    sc_norm  = normalizza(totals.to_dict(), mu_e)
    dom_norm = emozione_dominante(sc_norm)

    df_scores = pd.DataFrame([
    {'emozione': e, 'S_e (raw)': totals[e], 'μ_e': mu_e[e], 'S_e_norm': sc_norm[e]}
    for e in BASIC_EMOTIONS
    ]).set_index('emozione').round(3)
    display(df_scores)

print(f'=> Emozione dominante (corpus_mean): {dom_norm.upper()}')

,S_e (raw),μ_e,S_e_norm
emozione,,,
gioia,24.764,6.992,3.542
aspettativa,28.516,8.343,3.418
rabbia,12.971,5.020,2.584
disgusto,9.343,3.561,2.624
tristezza,13.091,5.099,2.567
sorpresa,20.471,6.731,3.042
paura,15.760,5.673,2.778
fiducia,25.139,7.231,3.476


=> Emozione dominante (corpus_mean): GIOIA


In [37]:
plot_radar_single(
        sc_norm, dom_norm,
        title=f'Profilo emotivo normalizzato — commento {CID} (metodo finale)',
        height=480,
    ).show()

## Confronto: score grezzo vs corpus_mean normalizzato

In [38]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    plot_raw_vs_norm_bars(totals.to_dict(), sc_norm, CID).show()

    print(f'Lemmi usati ({len(lemmi_usati)}): {lemmi_usati}')
    print(f'Emozione dominante raw          : {dom_raw.upper()}  (score: {totals[dom_raw]:.3f})')
    print(f'Emozione dominante corpus_mean  : {dom_norm.upper()}  (score: {sc_norm[dom_norm]:.3f})')

Lemmi usati (44): ['esperienza', 'avere', 'smettere', 'fare', 'sport', 'anno', 'arrivare', 'punto', 'odiare', 'fisico', 'avere', 'iniziare', 'palestra', 'amico', 'avere', 'leggere', 'palestra', 'non', 'fare', 'importante', 'inizio', 'difficile', 'vedere', 'paio', 'mese', 'risultare', 'normale', 'consigliare', 'non', 'trovare', 'sempre', 'motivo', 'non', 'iniziare', 'piacere', 'buttare', 'soddisfazione', 'dare', 'vedere', 'risultato', 'buona', 'fortuna', 'forte', '💪']
Emozione dominante raw          : ASPETTATIVA  (score: 28.516)
Emozione dominante corpus_mean  : GIOIA  (score: 3.542)
